In [3]:
# script_4_test_final.py
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
from skimage import io, transform
from tqdm import tqdm
from mmdet.apis import init_detector, inference_detector
from mmdet.utils import register_all_modules
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference

def calculate_dice(mask_true, mask_pred):
    m_true, m_pred = np.asarray(mask_true).astype(bool), np.asarray(mask_pred).astype(bool)
    if m_true.sum() + m_pred.sum() == 0: return 1.0
    return 2 * np.logical_and(m_true, m_pred).sum() / (m_true.sum() + m_pred.sum())

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Appareil detecte : {device}")
register_all_modules()

# --- Chargement de la configuration calibrée ---
try:
    with open("optimal_threshold.txt", "r") as f:
        OPTIMAL_THRESH = float(f.read().strip())
    print(f"Seuil de decision charge : {OPTIMAL_THRESH:.2f}")
except:
    print("Fichier optimal_threshold.txt introuvable. Seuil force a 0.25.")
    OPTIMAL_THRESH = 0.25

# --- Modèles ---
dino_model = init_detector('work_dirs/tumor_config_v3/tumor_config_v3.py', 
                           'work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth', device=device)

ensemble_models = []
for i in range(1, 6):
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 2))
    model.load_state_dict(torch.load(f"resnet_fold_{i}.pth", map_location=device))
    model = model.to(device).eval()
    ensemble_models.append(model)

crop_transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.5]*3, [0.5]*3)])

medsam_model = sam_model_registry["vit_b"](checkpoint="MedSAM/work_dir/MedSAM/medsam_vit_b.pth").to(device).eval()

# --- Données ---
base_dir = "data/MSD_pancreas"
with open(os.path.join(base_dir, "test.json"), 'r') as f:
    test_images_list = json.load(f)['images']

results_list = []
print(f"\nDebut de l'evaluation finale sur les {len(test_images_list)} images...")

for img_info in tqdm(test_images_list, desc="Inference Test"):
    file_name = img_info['file_name']
    img_path = os.path.join(base_dir, file_name)
    mask_path = os.path.join(base_dir, file_name.replace("/images/", "/masks/"))

    if not os.path.exists(img_path) or not os.path.exists(mask_path): continue
        
    img_3c = np.repeat(io.imread(img_path)[:, :, None], 3, axis=-1) if len(io.imread(img_path).shape) == 2 else io.imread(img_path)
    H, W, _ = img_3c.shape

    true_seg = np.zeros_like(io.imread(mask_path), dtype=np.uint8)
    true_seg[io.imread(mask_path) == 2] = 1
    has_tumor = true_seg.sum() > 0

    full_medsam_seg = np.zeros((H, W), dtype=np.uint8)
    prob_tumor = 0.0 

    with torch.no_grad():
        result = inference_detector(dino_model, img_path, text_prompt="pancreas . tumor .")
        scores, bboxes, labels = result.pred_instances.scores.cpu().numpy(), result.pred_instances.bboxes.cpu().numpy(), result.pred_instances.labels.cpu().numpy()
        
        pancreas_mask, tumor_mask = (labels == 0) & (scores > 0.3), (labels == 1) & (scores > 0.05)    
        pancreas_boxes, pancreas_scores = bboxes[pancreas_mask], scores[pancreas_mask]
        tumor_boxes, tumor_scores = bboxes[tumor_mask], scores[tumor_mask]

        best_tumor_box = None

        if len(pancreas_boxes) > 0 and len(tumor_boxes) > 0:
            p_x1, p_y1, p_x2, p_y2 = pancreas_boxes[np.argmax(pancreas_scores)]
            p_x1, p_y1, p_x2, p_y2 = max(0, p_x1 - 20), max(0, p_y1 - 20), p_x2 + 20, p_y2 + 20

            valid_tumor_boxes, valid_tumor_scores = [], []
            for t_box, t_score in zip(tumor_boxes, tumor_scores):
                inter_x1, inter_y1 = max(t_box[0], p_x1), max(t_box[1], p_y1)
                inter_x2, inter_y2 = min(t_box[2], p_x2), min(t_box[3], p_y2)
                if inter_x2 > inter_x1 and inter_y2 > inter_y1:
                    if ((inter_x2 - inter_x1) * (inter_y2 - inter_y1)) / ((t_box[2] - t_box[0]) * (t_box[3] - t_box[1])) >= 0.1:
                        valid_tumor_boxes.append(t_box)
                        valid_tumor_scores.append(t_score)
            
            if valid_tumor_boxes: best_tumor_box = valid_tumor_boxes[np.argmax(valid_tumor_scores)]
                
        elif len(pancreas_boxes) == 0 and len(tumor_boxes) > 0:
            best_tumor_box = tumor_boxes[np.argmax(tumor_scores)]

        # Filtre géométrique
        if best_tumor_box is not None:
            x1, y1, x2, y2 = map(int, best_tumor_box)
            if not (100 <= (x2 - x1) * (y2 - y1) <= 15000):
                best_tumor_box = None

        if best_tumor_box is not None:
            x1, y1, x2, y2 = map(int, best_tumor_box)
            c_x1, c_y1 = max(0, x1 - 5), max(0, y1 - 5)
            c_x2, c_y2 = min(W, x2 + 5), min(H, y2 + 5)
            crop_np = img_3c[c_y1:c_y2, c_x1:c_x2]
            
            if crop_np.shape[0] > 0 and crop_np.shape[1] > 0:
                crop_tensor = crop_transform(Image.fromarray(crop_np)).unsqueeze(0).to(device)
                prob_tumor = sum(torch.nn.functional.softmax(model(crop_tensor), dim=1)[0][1].item() for model in ensemble_models) / 5.0 

        # Décision stricte basée sur le seuil calibré
        if best_tumor_box is not None and prob_tumor >= OPTIMAL_THRESH:
            img_1024 = transform.resize(img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True).astype(np.uint8)
            img_1024 = (img_1024 - img_1024.min()) / np.clip(img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None)
            image_embedding = medsam_model.image_encoder(torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device))
            
            box_1024 = np.array([best_tumor_box]) / np.array([W, H, W, H]) * 1024
            full_medsam_seg[medsam_inference(medsam_model, image_embedding, box_1024, H, W) > 0] = 1
            final_dice = calculate_dice(true_seg, full_medsam_seg)
        else:
            final_dice = 0.0 if has_tumor else 1.0

    # Sauvegarde des métriques de tracking
    results_list.append({
        "file_name": file_name, 
        "has_tumor": has_tumor, 
        "final_dice": final_dice,
        "dino_proposed_box": best_tumor_box is not None,
        "resnet_prob": prob_tumor
    })

# --- Bilan ---
df = pd.DataFrame(results_list)
df_tumor, df_no_tumor = df[df['has_tumor'] == True], df[df['has_tumor'] == False]

print("\n" + "="*50)
print(f"RÉSULTATS FINAUX (Seuil pré-calibré = {OPTIMAL_THRESH:.2f})")
print("="*50)
print(f"DICE MOYEN GLOBAL : {df['final_dice'].mean():.4f}")
print(f"DICE MEDIAN GLOBAL : {df['final_dice'].median():.4f}")

if not df_tumor.empty:
    print("\n--- SCANS AVEC TUMEUR ---")
    print(f"DICE MOYEN : {df_tumor['final_dice'].mean():.4f}")
    df_fn = df_tumor[df_tumor['final_dice'] == 0.0]
    fn_count = len(df_fn)
    print(f"\nFAUX NÉGATIFS TOTAUX : {fn_count} / {len(df_tumor)} ({(fn_count/len(df_tumor))*100:.1f}%)")
    
    if fn_count > 0:
        fn_dino = len(df_fn[df_fn['dino_proposed_box'] == False])
        fn_resnet = len(df_fn[(df_fn['dino_proposed_box'] == True) & (df_fn['resnet_prob'] < OPTIMAL_THRESH)])
        fn_medsam = len(df_fn[(df_fn['dino_proposed_box'] == True) & (df_fn['resnet_prob'] >= OPTIMAL_THRESH)])
        
        print(f"  -> Causés par DINO (Aucune boîte valide) : {fn_dino}")
        print(f"  -> Causés par ResNet (Score rejeté < {OPTIMAL_THRESH:.2f}) : {fn_resnet}")
        print(f"  -> Causés par MedSAM (Mauvaise cible segmentée) : {fn_medsam}")

if not df_no_tumor.empty:
    print("\n--- SCANS SANS TUMEUR ---")
    print(f"DICE MOYEN : {df_no_tumor['final_dice'].mean():.4f}")
    fp_count = len(df_no_tumor[df_no_tumor['final_dice'] == 0.0])
    print(f"Faux Positifs : {fp_count} / {len(df_no_tumor)}")
print("="*50)

Appareil detecte : cuda
Seuil de decision charge : 0.37


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loads checkpoint by local backend from path: work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth


/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filena


Debut de l'evaluation finale sur les 100 images...


...

noun_phrases: ['pancreas', 'tumor']


[nltk_data] Downloading package punkt to ~/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ~/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/home/paulb/INF8225/Projet/.venv/lib/python3.11/site-packages/mmcv/cnn/bricks/transformer.py:524: UserWarning: position encoding of key ismissing in MultiheadAttention.
  warnings.warn(f'position encoding of key is'


noun_phrases: ['pancreas', 'tumor']


Inference Test: 100%|██████████| 100/100 [01:16<00:00,  1.30it/s]


RÉSULTATS FINAUX (Seuil pré-calibré = 0.37)
DICE MOYEN GLOBAL : 0.5916
DICE MEDIAN GLOBAL : 0.7795

--- SCANS AVEC TUMEUR ---
DICE MOYEN : 0.5232

FAUX NÉGATIFS TOTAUX : 14 / 50 (28.0%)
  -> Causés par DINO (Aucune boîte valide) : 0
  -> Causés par ResNet (Score rejeté < 0.37) : 14
  -> Causés par MedSAM (Mauvaise cible segmentée) : 0

--- SCANS SANS TUMEUR ---
DICE MOYEN : 0.6600
Faux Positifs : 17 / 50
